# LoRA и полное обучение

Что обучается в адаптере и почему экономия памяти не равна размеру весов.

> Все примеры этого урока работают на CPU, не скачивают модели и используют искусственные данные. Они объясняют механизм; качество настоящего обучения проверяется отдельно.

## Что важно понять

LoRA добавляет к замороженной матрице W низкоранговое обновление B×A. Обычно A инициализируют случайно, B — нулями: стартовая функция остаётся прежней, а градиент B уже ненулевой. Масштаб alpha/r влияет на обновление.

Заморозка уменьшает память градиентов и оптимизатора, но базовые веса и активации остаются. QLoRA дополнительно квантует базу и требует совместимого backend. Этот CPU-пример показывает LoRA без квантизации и Unsloth.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Запустите эксперимент

Запускайте ячейки по порядку. Код можно менять; Shift + Enter выполняет текущую ячейку.

In [ ]:
base = nn.Linear(12, 8); base.requires_grad_(False)
rank, alpha = 2, 4
A = nn.Parameter(torch.randn(rank, 12) * 0.02)
B = nn.Parameter(torch.zeros(8, rank))
optimizer = torch.optim.AdamW([A, B], lr=0.03)
x = torch.randn(32, 12); target = torch.randn(32, 8)
frozen = base.weight.detach().clone()
history = []
for step in range(50):
    prediction = base(x) + (x @ A.T @ B.T) * alpha / rank
    loss = F.mse_loss(prediction, target)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
    history.append(loss.item())
assert torch.equal(base.weight, frozen)
print('trainable:', A.numel() + B.numel(), 'base:', sum(p.numel() for p in base.parameters()))
plt.plot(history, color='#d73535'); plt.ylabel('LoRA MSE'); plt.show()

## Попробуйте сами

Поставьте rank=1 и rank=8, повторив ячейку с начала. Сравните число параметров и итоговую ошибку. Что произойдёт, если и A, и B изначально нулевые?

Сохраните копию ноутбука и запишите, что изменили, чего ожидали и что измерили.

## Продолжить в студии

Кнопка «Открыть раздел студии» над уроком ведёт к связанным инструментам. Учебный код сам не запускает большие тренировки. Перед реальным запуском проверьте данные, ресурсы и поддержку метода у выбранной модели.

## Источники

- [LoRA](https://arxiv.org/abs/2106.09685)
- [QLoRA](https://arxiv.org/abs/2305.14314)